# Intro to Cognee - building a Personal Memory Assistant

## Why personal memory?

Personal attention is fragmented by default. A tiny obligation can arrive in a Slack mention, a family chat, an email thread, or a follow-up buried under a reply. The hard part is not just storing the messages. The hard part is noticing which ones still expect something from you.

A personal memory assistant should answer questions like:

- Who tagged me and still needs a reply?
- What did I already handle?
- What did I answer, but still need to finish?
- What is due by EOD, EOW, or a specific posted deadline?

This notebook builds that foundation with Cognee. We ingest a small personal corpus for Veljko: Slack-style channel exports and email-thread exports. Then we ask the graph for an EOD/EOW reminder brief.

The notebook follows a simple memory-assistant loop:

1. **Load data** - `remember()` ingests text and builds the graph in one call
2. **Recall** - ask natural-language questions over the personal memory
3. **Node sets** - tag Slack and email sources for scoping
4. **Custom graph model** - extract typed obligations and response state
5. **Add another source** - email joins the same graph as Slack
6. **Life-assistant brief** - ask for missed replies, handled items, and open follow-up


## The API (Cognee 1.x)

The modern surface is three async functions:

```python
await cognee.remember(text, dataset_name=...)   # 1. ingest + build the graph
await cognee.recall(query)                       # 2. ask questions
await cognee.improve(dataset=...)                # 3. refine the graph from feedback/use
```

`remember()` replaces the old two-step `add()` + `cognify()` flow for everyday use. It ingests the text, extracts graph structure, and runs the default self-improvement pass.

For a personal assistant, that means every normalized source - Slack, email, notes, calendar summaries - can become memory with the same call. The source matters less than the shape: clear speaker, timestamp, and message text.


## Setup

The setup cell below puts a placeholder `LLM_API_KEY` into `os.environ`. Replace `"your-api-key"` with a real key before running the notebook.

> Run once from the repo root: `uv pip install -e . jupyterlab ipykernel`


In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
os.environ.setdefault("LLM_API_KEY", "your-api-key")
os.environ.setdefault("CACHING", "true")

import cognee
from cognee.modules.users.methods import get_default_user
from cognee.modules.data.methods import get_authorized_existing_datasets
from cognee.context_global_variables import set_database_global_context_variables

# Works whether the kernel starts in the repo root or a notebook subdirectory.
ROOT = Path.cwd()
REPO = ROOT if (ROOT / "sample_data").exists() else ROOT.parent
DATA = REPO / "sample_data" / "personal_memory"
DATASET = "personal_memory"
TARGET_PERSON = "veljko@topoteretes.com"
TARGET_MENTION = "@veljko@topoteretes.com"

async def reset():
    """Wipe local Cognee state so the notebook can be re-run cleanly."""
    await cognee.prune.prune_data()
    await cognee.prune.prune_system(metadata=True)

def answer(results):
    """Pull a readable answer out of a recall() result list."""
    for r in results:
        for attr in ("text", "answer", "content"):
            value = getattr(r, attr, None)
            if value:
                return value
    return str(results[0]) if results else "<no answer>"

async def show_graph(filename, dataset, height=760):
    """Render a dataset-scoped graph to HTML, with open/download controls."""
    user = await get_default_user()
    datasets = await get_authorized_existing_datasets([dataset], "read", user)

    if not datasets:
        raise ValueError(f"Dataset not found or not readable: {dataset}")

    dataset_obj = datasets[0]
    owner_id = getattr(dataset_obj, "owner_id", None) or user.id

    out = ROOT / filename
    async with set_database_global_context_variables(dataset_obj.id, owner_id):
        await cognee.visualize_graph(str(out))

    from html import escape
    from IPython.display import HTML, IFrame, display

    href = escape(filename, quote=True)
    controls = (
        '<div style="display:flex;gap:8px;margin:8px 0 10px;align-items:center">'
        f'<a href="{href}" target="_blank" rel="noopener" '
        'style="padding:6px 10px;border:1px solid #ccc;border-radius:6px;text-decoration:none">'
        'Open full screen</a>'
        f'<a href="{href}" download '
        'style="padding:6px 10px;border:1px solid #ccc;border-radius:6px;text-decoration:none">'
        'Download HTML</a>'
        '</div>'
    )
    display(HTML(controls))
    return IFrame(src=filename, width="100%", height=height)


print("Data directory:", DATA)
print("Slack files:", [p.name for p in sorted(DATA.glob("slack_*.txt"))])
print("Email files:", [p.name for p in sorted(DATA.glob("email_*.txt"))])


2026-06-11T09:20:46.946015 [info     ] Deleted old log file: /Users/milenko/.cognee/logs/2026-06-11_09-38-41.log [cognee.shared.logging_utils]

2026-06-11T09:20:46.946382 [info     ] Log file created at: /Users/milenko/.cognee/logs/2026-06-11_11-20-46.log [cognee.shared.logging_utils] log_file=/Users/milenko/.cognee/logs/2026-06-11_11-20-46.log

2026-06-11T09:20:46.946644 [warning  ] Cognee 1.0 changes: New API — remember/recall/forget/improve (V1 add/cognify/search still work). Session memory enabled by default (CACHING=false to disable). Multi-user access control on by default (ENABLE_BACKEND_ACCESS_CONTROL=false to disable). Agents (@cognee.agent) auto-verified on registration. See https://docs.cognee.ai/ [cognee.shared.logging_utils]

2026-06-11T09:20:46.946894 [info     ] Logging initialized            [cognee.shared.logging_utils] cognee_version=1.1.2 database_path=/Users/milenko/cognee/cognee-companybrain-hackathon/.venv/lib/python3.13/site-packages/cognee/.cognee_system/databa

Data directory: /Users/milenko/cognee/cognee-companybrain-hackathon/sample_data/personal_memory
Slack files: ['slack_design.txt', 'slack_finance.txt', 'slack_health.txt', 'slack_product.txt', 'slack_social.txt']
Email files: ['email_landlord_lease_renewal.txt', 'email_parent_evening.txt', 'email_trip_documents.txt']


## 1. Load Slack messages with `remember()`

The Slack files use a lightweight export format: one channel transcript per file, with one line per message.

```text
# #channel-name: short thread title
[person@example.com, 2026-06-11T09:12] message text
```

In a real assistant, these would come from the Slack API. For the notebook, plain text keeps the data easy to inspect.


In [2]:
slack_files = sorted(DATA.glob("slack_*.txt"))
print(slack_files[0].read_text())


# #design-studio: Portfolio screenshots for Friday demo
[mina@topoteretes.com, 2026-06-11T09:12] @veljko@topoteretes.com can you upload three screenshots from the new memory graph flow before EOD 2026-06-11? I need them for the Friday demo deck.
[nikola@topoteretes.com, 2026-06-11T09:18] The deck is in Drive under Demo / June / Memory graph.
[mina@topoteretes.com, 2026-06-11T11:03] Reminder that the screenshots should show the inbox item, the extracted obligation, and the final reminder card.
[nikola@topoteretes.com, 2026-06-11T14:26] I updated slide 4, but I still do not have Veljko's screenshots.



In [3]:
await reset()

slack_files = sorted(DATA.glob("slack_*.txt"))
for f in slack_files:
    result = await cognee.remember(
        f.read_text(),
        dataset_name=DATASET,
        node_set=["source:slack", f"file:{f.stem}"],
    )
    print("remembered", f.name, "- status:", result.to_dict().get("status"))



2026-06-11T09:20:50.626626 [info     ] Database deleted successfully. [cognee.shared.logging_utils]

Storage manager absolute path: /Users/milenko/cognee/cognee-companybrain-hackathon/.venv/lib/python3.13/site-packages/cognee/.cognee_cache

Deleting cache...             

✓ Cache deleted successfully! 

Skipping vector startup migrations. Could not access dataset_database table: (sqlite3.OperationalError) no such table: dataset_database
[SQL: SELECT dataset_database.owner_id, dataset_database.dataset_id, dataset_database.vector_database_name, dataset_database.graph_database_name, dataset_database.vector_database_provider, dataset_database.graph_database_provider, dataset_database.graph_dataset_database_handler, dataset_database.vector_dataset_database_handler, dataset_database.vector_database_url, dataset_database.graph_database_url, dataset_database.vector_database_key, dataset_database.graph_database_key, dataset_database.graph_database_connection_info, dataset_database.vector_datab

remembered slack_design.txt - status: completed



2026-06-11T09:21:52.363547 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-11T09:21:52.364273 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-11T09:21:52.419577 [info     ] Pipeline run started: `2051bba8-8761-5cc6-94bb-01b54685cf02` [run_tasks_with_telemetry()]

2026-06-11T09:21:52.425907 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2026-06-11T09:21:52.432224 [info     ] Async Generator task started: `extract_chunks_from_documents` [run_tasks_base]

2026-06-11T09:21:52.441013 [info     ] Coroutine task started: `extract_graph_and_summarize` [run_tasks_base]

2026-06-11T09:22:31.375816 [info     ] No close match found for 'person' in category 'classes' [OntologyAdapter]

2026-06-11T09:22:31.376397 [info     ] No close match found for 'tara@topoteretes.com' in category 'individuals' [OntologyAdapter]

2026-06-11T09:22:31.376728 [info     ] No close matc

remembered slack_finance.txt - status: completed



2026-06-11T09:22:34.143795 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-11T09:22:34.144490 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-11T09:22:34.200574 [info     ] Pipeline run started: `2051bba8-8761-5cc6-94bb-01b54685cf02` [run_tasks_with_telemetry()]

2026-06-11T09:22:34.207100 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2026-06-11T09:22:34.215572 [info     ] Async Generator task started: `extract_chunks_from_documents` [run_tasks_base]

2026-06-11T09:22:34.225866 [info     ] Coroutine task started: `extract_graph_and_summarize` [run_tasks_base]

2026-06-11T09:23:23.562387 [info     ] No close match found for 'person' in category 'classes' [OntologyAdapter]

2026-06-11T09:23:23.562958 [info     ] No close match found for 'dragan' in category 'individuals' [OntologyAdapter]

2026-06-11T09:23:23.563301 [info     ] No close match found for 'v

remembered slack_health.txt - status: completed



2026-06-11T09:23:26.492456 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-11T09:23:26.493186 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-11T09:23:26.548913 [info     ] Pipeline run started: `2051bba8-8761-5cc6-94bb-01b54685cf02` [run_tasks_with_telemetry()]

2026-06-11T09:23:26.555275 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2026-06-11T09:23:26.561528 [info     ] Async Generator task started: `extract_chunks_from_documents` [run_tasks_base]

2026-06-11T09:23:26.570607 [info     ] Coroutine task started: `extract_graph_and_summarize` [run_tasks_base]

2026-06-11T09:24:21.067336 [info     ] No close match found for 'person' in category 'classes' [OntologyAdapter]

2026-06-11T09:24:21.067920 [info     ] No close match found for 'ana@topoteretes.com' in category 'individuals' [OntologyAdapter]

2026-06-11T09:24:21.068240 [info     ] No close match

remembered slack_product.txt - status: completed



2026-06-11T09:24:23.925188 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-11T09:24:23.925920 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-11T09:24:23.984132 [info     ] Pipeline run started: `2051bba8-8761-5cc6-94bb-01b54685cf02` [run_tasks_with_telemetry()]

2026-06-11T09:24:23.990480 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2026-06-11T09:24:23.996736 [info     ] Async Generator task started: `extract_chunks_from_documents` [run_tasks_base]

2026-06-11T09:24:24.005534 [info     ] Coroutine task started: `extract_graph_and_summarize` [run_tasks_base]

2026-06-11T09:25:04.403810 [info     ] No close match found for 'person' in category 'classes' [OntologyAdapter]

2026-06-11T09:25:04.404437 [info     ] No close match found for 'luka (luka@example.com)' in category 'individuals' [OntologyAdapter]

2026-06-11T09:25:04.404817 [info     ] No close m

remembered slack_social.txt - status: completed


### Graph: Slack-only default extraction

This renders the first dataset while it still exists. The next section resets local state to keep the node-set demo isolated.

In [4]:
await show_graph("personal_memory_slack_default_graph.html", DATASET)



2026-06-11T09:25:07.842663 [info     ] Retrieved 73 nodes and 228 edges in 0.01 seconds [cognee.shared.logging_utils]

2026-06-11T09:25:07.847163 [info     ] Graph visualization saved as /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/personal_memory_slack_default_graph.html [cognee.shared.logging_utils]

2026-06-11T09:25:07.847458 [info     ] The HTML file has been stored at path: /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/personal_memory_slack_default_graph.html [cognee.shared.logging_utils]

2026-06-11T09:25:07.875526 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]


## 2. Recall - ask for missed replies

Now ask the graph like a personal assistant. The key distinction is not just "was Veljko mentioned?" but whether the thread shows a response or completion.


In [5]:
results = await cognee.recall(
    "Which Slack messages tagged Veljko and still need his response or action by EOD 2026-06-11? Separate missed replies from items where he replied but more work is needed.",
    datasets=[DATASET],
)
print(answer(results))



2026-06-11T09:25:07.890927 [info     ] query_router: routed=TEMPORAL score=3.0 query='Which Slack messages tagged Veljko and still need his response or action by EOD 2026-06-11? Separate missed replies from items where he replied but more work is needed.' scores={'TEMPORAL': 3.0} [query_router]

2026-06-11T09:25:15.346816 [info     ] No events identified based on timestamp filtering, performing retrieval using triplet search on events and entities. [cognee.shared.logging_utils]

2026-06-11T09:25:16.499853 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 1.15s [cognee.shared.logging_utils]

2026-06-11T09:25:16.500483 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-11T09:25:16.507011 [info     ] ID-filtered retrieval: 73 nodes and 228 edges in 0.01s [cognee.shared.logging_utils]

2026-06-11T09:25:16.508332 [info     ] Graph projection completed: 73 nodes, 228 edges in 0.00s [CogneeGraph]

2026-06-11T09:25:16.5092

Missed replies (no reply/action from Veljko recorded by EOD 2026-06-11):
1) Portfolio screenshots for Friday demo — Mina asked 2026-06-11T09:12: “can you upload three screenshots from the new memory graph flow before EOD 2026-06-11?” Reminder at 2026-06-11T11:03 about which three shots to include. Nikola (2026-06-11T14:26) reports he still does not have Veljko’s screenshots.
2) Saturday climbing plan — Luka asked 2026-06-11T07:55: “are you in for climbing on Saturday? Please answer by EOD 2026-06-11 so I can reserve the lanes.” No reply from Veljko recorded.

Replied but more work needed (Veljko replied but did not complete the required action by the deadline):
1) Dentist insurance form — Dragan told Veljko 2026-06-10T16:42 the dentist needs the insurance form before the appointment; Veljko replied 2026-06-10T17:05 “I will fill it out tonight.” Follow-up 2026-06-11T09:14 reminded form must be in their inbox before 2026-06-11 17:00; Sofia at 2026-06-11T13:20 reports the submitted form i

In [6]:
# Peek at the retrieved context behind the reminder answer.
ctx = await cognee.recall(
    "Veljko tagged EOD missed response open action",
    datasets=[DATASET],
    only_context=True,
    top_k=5,
)
for c in ctx:
    print("-", str(getattr(c, "content", c))[:300])



2026-06-11T09:25:33.006475 [info     ] query_router: no patterns matched, default=GRAPH_COMPLETION query='Veljko tagged EOD missed response open action' [query_router]

2026-06-11T09:25:34.057691 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 0.95s [cognee.shared.logging_utils]

2026-06-11T09:25:34.058323 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-11T09:25:34.065562 [info     ] ID-filtered retrieval: 73 nodes and 228 edges in 0.01s [cognee.shared.logging_utils]

2026-06-11T09:25:34.066912 [info     ] Graph projection completed: 73 nodes, 228 edges in 0.00s [CogneeGraph]

2026-06-11T09:25:34.067723 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 6, 'connection_count': 5}

2026-06-11T09:25:34.094905 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-11T09:25:34.132171 [info     ] recall: 1 results across sources=['graph']

- kind='graph_completion' search_type='GRAPH_COMPLETION' text="Nodes:\nNode: # #design-studio: Portfolio screenshots for Friday demo... [screenshots, demo, i]\n__node_content_start__\n# #design-studio: Portfolio screenshots for Friday demo\n[mina@topoteretes.com, 2026-06-11T09:12] @veljko@topoteretes.


## 3. Node sets - separate Slack from email

Node sets are tags attached at ingest time. Here we tag each source by channel/file and by broader source type, so the assistant can later answer from only Slack, only email, or the whole personal memory.

```python
await cognee.remember(text, dataset_name=DATASET, node_set=["source:slack"])
await cognee.remember(text, dataset_name=DATASET, node_set=["source:email"])
```


In [7]:
NS_DATASET = "personal_memory_node_sets"
await reset()

for f in sorted(DATA.glob("slack_*.txt")):
    await cognee.remember(f.read_text(), dataset_name=NS_DATASET, node_set=["source:slack", f"file:{f.stem}"])
for f in sorted(DATA.glob("email_*.txt")):
    await cognee.remember(f.read_text(), dataset_name=NS_DATASET, node_set=["source:email", f"file:{f.stem}"])

print("tagged Slack and email into separate node sets")



2026-06-11T09:25:34.311787 [info     ] Deleted Ladybug database files at /Users/milenko/cognee/cognee-companybrain-hackathon/.venv/lib/python3.13/site-packages/cognee/.cognee_system/databases/2fb0015f-001d-4133-a8f3-02d80e9a1c11/7b572124-84a6-58ce-b26c-f7744b895da3.lbug [cognee.shared.logging_utils]

2026-06-11T09:25:34.319563 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-11T09:25:36.795511 [info     ] Database deleted successfully. [cognee.shared.logging_utils]

Deleting cache...             

✓ Cache deleted successfully! 

User 9285a396-e0e8-4a74-be15-49914998348d has registered.

2026-06-11T09:25:36.935438 [info     ] Pipeline run started: `44e945ac-8842-5ce6-bf52-3cf17c34c7e4` [run_tasks_with_telemetry()]

2026-06-11T09:25:36.941627 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-06-11T09:25:36.947786 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-06-11T09:25:36.963318 [info 

tagged Slack and email into separate node sets


In [8]:
slack_only = await cognee.recall(
    "What Slack obligations are still open for Veljko?",
    datasets=[NS_DATASET],
    node_name=["source:slack"],
    scope=["graph"],
)
print(answer(slack_only))



2026-06-11T09:32:49.818721 [info     ] query_router: no patterns matched, default=GRAPH_COMPLETION query='What Slack obligations are still open for Veljko?' [query_router]

2026-06-11T09:32:50.695633 [info     ] Vector collection retrieval completed: Retrieved distances from 3 collections in 0.77s [cognee.shared.logging_utils]

2026-06-11T09:32:50.696151 [info     ] Retrieving graph filtered by node type and node name (NodeSet). [CogneeGraph]

2026-06-11T09:32:50.710536 [info     ] Graph projection completed: 53 nodes, 322 edges in 0.00s [CogneeGraph]

2026-06-11T09:32:50.711376 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 6, 'connection_count': 10}

2026-06-11T09:32:57.974244 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-11T09:32:58.029118 [info     ] recall: 1 results across sources=['graph'] (session=-) [recall]


Open Slack obligations for Veljko:
- Upload three screenshots from the new memory-graph flow (inbox item, extracted obligation, final reminder card) for the Friday demo — requested by Mina; due EOD 2026-06-11 (Nikola still missing them).
- Reply to Luka’s Saturday climbing invite (confirm whether he’s in) — requested by Luka; reply due EOD 2026-06-11.


### Graph: Slack and email node-set dataset

This dataset is tagged by source (`source:slack`, `source:email`) and by file. It shows how the same corpus can be organized for source scoping.

In [9]:
await show_graph("personal_memory_node_sets_graph.html", NS_DATASET)



2026-06-11T09:32:58.133927 [info     ] Retrieved 118 nodes and 407 edges in 0.01 seconds [cognee.shared.logging_utils]

2026-06-11T09:32:58.139357 [info     ] Graph visualization saved as /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/personal_memory_node_sets_graph.html [cognee.shared.logging_utils]

2026-06-11T09:32:58.139658 [info     ] The HTML file has been stored at path: /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/personal_memory_node_sets_graph.html [cognee.shared.logging_utils]

2026-06-11T09:32:58.166686 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]


## 4. Custom graph model - extract obligations

Default extraction can answer questions, but a life assistant benefits from a typed schema. We care about people, messages, and obligations:

- who asked
- who owns the next action
- what the deadline is
- whether the state is `missed_response`, `open_followup`, or `handled`

The custom graph model below defines those concepts directly as `Person`, `Message`, `Obligation`, and `MemoryThread`. The graph visualization in this section should therefore contain these newly defined model nodes, rather than only Cognee's default `Entity` / `EntityType` nodes.

The extraction prompt is intentionally strict: acknowledging a request is not the same as completing it.


In [10]:
from typing import Optional
from cognee.low_level import DataPoint

class Person(DataPoint):
    email: str
    name: str
    metadata: dict = {"index_fields": ["email", "name"], "identity_fields": ["email"]}

class Obligation(DataPoint):
    summary: str
    owner: Person
    requested_by: Person
    deadline: str
    status: str
    next_action: str
    evidence: str
    source: str
    metadata: dict = {"index_fields": ["summary", "deadline", "status", "next_action"]}

class Message(DataPoint):
    speaker: Person
    text: str
    sent_at: str
    mentions: Optional[list[Person]] = None
    obligations: Optional[list[Obligation]] = None
    metadata: dict = {"index_fields": ["text", "sent_at"]}

class MemoryThread(DataPoint):
    source: str
    title: str
    participants: list[Person]
    messages: list[Message]
    obligations: Optional[list[Obligation]] = None
    metadata: dict = {"index_fields": ["source", "title"]}


In [11]:
EXTRACTION_PROMPT = f"""Extract a personal memory graph from this Slack or email thread.

Target person: {TARGET_PERSON}

Use the provided MemoryThread graph model with Person, Message, and Obligation nodes.
Keep source evidence precise and short.

For every request, question, or deadline directed at the target person, create an Obligation.
Set status as one of:
- missed_response: the target person was asked or tagged and no later target-person reply appears.
- open_followup: the target person replied or acknowledged, but the text does not show completion, or a later message says the work is still missing.
- handled: the target person replied and the thread shows the request was completed or no more action is needed.

Deadline rules:
- Preserve explicit dates and times when present.
- Preserve EOD and EOW wording together with any stated weekday or date.
- Preserve explicit dates such as Thursday 2026-06-11 and Friday 2026-06-12.

Do not mark acknowledgement as handled unless completion is explicit.
Prefer exact source text over summaries."""

TYPED_DATASET = "personal_memory_typed"
await reset()

for f in sorted(DATA.glob("slack_*.txt")):
    await cognee.remember(
        f.read_text(),
        dataset_name=TYPED_DATASET,
        node_set=["source:slack", f"file:{f.stem}"],
        graph_model=MemoryThread,
        custom_prompt=EXTRACTION_PROMPT,
    )

print("typed Slack memory graph built")



2026-06-11T09:32:58.332366 [info     ] Deleted Ladybug database files at /Users/milenko/cognee/cognee-companybrain-hackathon/.venv/lib/python3.13/site-packages/cognee/.cognee_system/databases/9285a396-e0e8-4a74-be15-49914998348d/7c832f3d-6213-5d23-a1c7-a4f3e44dbbc6.lbug [cognee.shared.logging_utils]

2026-06-11T09:32:58.340413 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-11T09:33:00.820183 [info     ] Database deleted successfully. [cognee.shared.logging_utils]

Deleting cache...             

✓ Cache deleted successfully! 

User dc6b87db-731b-4fee-9d54-b555508165fe has registered.

2026-06-11T09:33:00.960669 [info     ] Pipeline run started: `79e81865-30d7-54d7-86eb-eec7b123810b` [run_tasks_with_telemetry()]

2026-06-11T09:33:00.966881 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-06-11T09:33:00.973078 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-06-11T09:33:00.987965 [info 

typed Slack memory graph built


### Graph: Custom obligation model

This graph is built with the custom `MemoryThread` schema. Look for nodes shaped by the notebook's model definitions, especially `Person`, `Message`, `Obligation`, and `MemoryThread`, instead of the generic default extraction types.

In [12]:
await show_graph("personal_memory_custom_model_slack_graph.html", TYPED_DATASET)



2026-06-11T09:38:26.083848 [info     ] Retrieved 67 nodes and 123 edges in 0.01 seconds [cognee.shared.logging_utils]

2026-06-11T09:38:26.086982 [info     ] Graph visualization saved as /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/personal_memory_custom_model_slack_graph.html [cognee.shared.logging_utils]

2026-06-11T09:38:26.087249 [info     ] The HTML file has been stored at path: /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/personal_memory_custom_model_slack_graph.html [cognee.shared.logging_utils]

2026-06-11T09:38:26.115580 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]


In [13]:
typed_slack = await cognee.recall(
    "List Veljko's Slack obligations grouped by status: missed_response, open_followup, handled. Include deadline and source evidence.",
    datasets=[TYPED_DATASET],
)
print(answer(typed_slack))



2026-06-11T09:38:26.128920 [info     ] query_router: no patterns matched, default=GRAPH_COMPLETION query="List Veljko's Slack obligations grouped by status: missed_response, open_followup, handled. Include deadline and source evidence." [query_router]

2026-06-11T09:38:27.102749 [info     ] Vector collection retrieval completed: Retrieved distances from 20 collections in 0.87s [cognee.shared.logging_utils]

2026-06-11T09:38:27.103387 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-11T09:38:27.109750 [info     ] ID-filtered retrieval: 67 nodes and 123 edges in 0.01s [cognee.shared.logging_utils]

2026-06-11T09:38:27.110639 [info     ] Graph projection completed: 67 nodes, 123 edges in 0.00s [CogneeGraph]

2026-06-11T09:38:27.111571 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 16, 'connection_count': 10}

2026-06-11T09:38:47.073383 [info     ] Ladybug database closed successfully [cognee.shared.logging_ut

missed_response:
- Portfolio screenshots for Friday demo — deadline: EOD 2026-06-11
  source evidence: Mina @ 2026-06-11T09:12 asked @veljko to "upload three screenshots ... before EOD 2026-06-11"; Nikola @ 2026-06-11T14:26: "I updated slide 4, but I still do not have Veljko's screenshots."
- Saturday climbing RSVP — deadline: EOD 2026-06-11
  source evidence: Luka @ 2026-06-11T07:55: "are you in for climbing on Saturday? Please answer by EOD 2026-06-11"; no Veljko reply recorded.

open_followup:
- Dentist insurance form submission — deadline: must be in dentist inbox before 2026-06-11 17:00
  source evidence: Dragan @ 2026-06-10T16:42: "They need the insurance form before the appointment on Friday, 2026-06-12 at 09:30." Veljko @ 2026-06-10T17:05: "I will fill it out tonight"; Dragan @ 2026-06-11T09:14: reminded the form must be in inbox before 2026-06-11 17:00; Sofia @ 2026-06-11T13:20: "I do not see the submitted form yet."

handled:
- Reimbursement receipt (add May coworking receipt

## 5. Add email threads to the same graph

Now add email as a second source. The format is different from Slack, but it is still plain text with people, timestamps, and messages. We use the same `remember()` call and the same typed schema.


In [14]:
email_files = sorted(DATA.glob("email_*.txt"))
print(email_files[0].read_text())


# Email thread: Lease renewal confirmation

From: Ana Petrovic <ana.petrovic@oakline-homes.example>
To: Veljko <veljko@topoteretes.com>
Date: 2026-06-09T08:18
Subject: Lease renewal paperwork due Friday

Hi Veljko,

We are preparing lease renewals for July. Could you confirm by Friday,
2026-06-12 at 12:00 whether you want to renew for another 12 months?

If yes, I will send the DocuSign packet. If you need different dates, please
reply with your preferred move-out or renewal term.

Ana

---

From: Veljko <veljko@topoteretes.com>
To: Ana Petrovic <ana.petrovic@oakline-homes.example>
Date: 2026-06-09T10:02
Subject: Re: Lease renewal paperwork due Friday

Hi Ana,

Yes, I want to renew for another 12 months. Please send the DocuSign packet.

Thanks,
Veljko

---

From: Ana Petrovic <ana.petrovic@oakline-homes.example>
To: Veljko <veljko@topoteretes.com>
Date: 2026-06-10T14:37
Subject: Re: Lease renewal paperwork due Friday

Thanks, Veljko. I sent the DocuSign packet just now. Please sign it

In [15]:
for f in sorted(DATA.glob("email_*.txt")):
    await cognee.remember(
        f.read_text(),
        dataset_name=TYPED_DATASET,
        node_set=["source:email", f"file:{f.stem}"],
        graph_model=MemoryThread,
        custom_prompt=EXTRACTION_PROMPT,
    )
    print("remembered", f.name)



2026-06-11T09:38:47.174654 [info     ] Pipeline run started: `79e81865-30d7-54d7-86eb-eec7b123810b` [run_tasks_with_telemetry()]

2026-06-11T09:38:47.180879 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-06-11T09:38:47.186969 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-06-11T09:38:47.204497 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]

2026-06-11T09:38:47.210618 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]

2026-06-11T09:38:47.216671 [info     ] Pipeline run completed: `79e81865-30d7-54d7-86eb-eec7b123810b` [run_tasks_with_telemetry()]

2026-06-11T09:38:47.337056 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-11T09:38:47.337731 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-11T09:38:47.394716 [info     ] Pipeline run started: `06e505ba-d09d-5b9b-a0ec-af

remembered email_landlord_lease_renewal.txt



2026-06-11T09:40:54.354765 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-11T09:40:54.355535 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-11T09:40:54.418481 [info     ] Pipeline run started: `06e505ba-d09d-5b9b-a0ec-af8b49f5f386` [run_tasks_with_telemetry()]

2026-06-11T09:40:54.424818 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2026-06-11T09:40:54.431016 [info     ] Async Generator task started: `extract_chunks_from_documents` [run_tasks_base]

2026-06-11T09:40:54.440069 [info     ] Coroutine task started: `extract_graph_and_summarize` [run_tasks_base]

2026-06-11T09:42:01.985023 [info     ] Coroutine task started: `add_data_points` [run_tasks_base]

2026-06-11T09:42:02.001525 [info     ] Completed graph extraction for DataPoint [cognee.shared.logging_utils] extra={'datapoint_id': '1a1ea6ca-7603-525b-bed4-ae30a099bb6c', 'nodes_extracted': 1, 'edg

remembered email_parent_evening.txt



2026-06-11T09:42:05.760278 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-11T09:42:05.761025 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-11T09:42:05.831017 [info     ] Pipeline run started: `06e505ba-d09d-5b9b-a0ec-af8b49f5f386` [run_tasks_with_telemetry()]

2026-06-11T09:42:05.837469 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2026-06-11T09:42:05.843814 [info     ] Async Generator task started: `extract_chunks_from_documents` [run_tasks_base]

2026-06-11T09:42:05.852757 [info     ] Coroutine task started: `extract_graph_and_summarize` [run_tasks_base]

2026-06-11T09:42:56.287295 [info     ] Coroutine task started: `add_data_points` [run_tasks_base]

2026-06-11T09:42:56.307181 [info     ] Completed graph extraction for DataPoint [cognee.shared.logging_utils] extra={'datapoint_id': '1a1ea6ca-7603-525b-bed4-ae30a099bb6c', 'nodes_extracted': 1, 'edg

remembered email_trip_documents.txt


## 6. Life-assistant brief

This is the payoff: one typed obligation graph spanning Slack and email. Ask it for a reminder brief that a personal assistant could send at the end of the day or end of the week.


In [16]:
brief = await cognee.recall(
    "Create Veljko's reminder brief for Thursday 2026-06-11. Include: (1) missed replies due by EOD, (2) items he answered but still needs to finish, (3) items already handled and safe to ignore. Include source and deadline for each item.",
    datasets=[TYPED_DATASET],
)
print(answer(brief))



2026-06-11T09:42:59.009145 [info     ] query_router: routed=TEMPORAL score=3.0 query="Create Veljko's reminder brief for Thursday 2026-06-11. Include: (1) missed replies due by EOD, (2) items he answered but still needs to finish, (3) items already handled and safe to ignore. Include source and deadline for each item." scores={'TEMPORAL': 3.0} [query_router]

2026-06-11T09:43:05.106150 [info     ] No events identified based on timestamp filtering, performing retrieval using triplet search on events and entities. [cognee.shared.logging_utils]

2026-06-11T09:43:06.008885 [info     ] Vector collection retrieval completed: Retrieved distances from 30 collections in 0.90s [cognee.shared.logging_utils]

2026-06-11T09:43:06.009625 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-11T09:43:06.015792 [info     ] ID-filtered retrieval: 102 nodes and 193 edges in 0.01s [cognee.shared.logging_utils]

2026-06-11T09:43:06.017144 [info     ] Graph projection completed: 1

Reminder brief — Thursday 2026-06-11

1) Missed replies due by EOD 2026-06-11
- Portfolio screenshots for Friday demo — deadline: EOD 2026-06-11
  source: Mina @2026-06-11T09:12 asked Veljko to “upload three screenshots ... before EOD 2026-06-11”; Nikola @2026-06-11T14:26: “I updated slide 4, but I still do not have Veljko's screenshots.”
- Friday dinner RSVP (Jelena) — deadline: EOD 2026-06-11
  source: Jelena Markovic email 2026-06-10T19:24: “Can you let me know by EOD Thursday, 2026-06-11 whether you and Milica are coming?”; Milica cc’d Veljko 2026-06-11T08:46 asking him to decide.
- Saturday climbing RSVP (Luka) — deadline: EOD 2026-06-11
  source: Luka @2026-06-11T07:55: “are you in for climbing on Saturday? Please answer by EOD 2026-06-11 so I can reserve the lanes.”

2) Items you answered but still need to finish
- Dentist insurance form submission — deadline: must be in dentist inbox before 2026-06-11 17:00
  source: Dragan @2026-06-10T16:42 (office needs form before appointmen

In [17]:
eow = await cognee.recall(
    "What should Veljko remember before EOW Friday 2026-06-12? Focus on unresolved obligations across Slack and email.",
    datasets=[TYPED_DATASET],
)
print(answer(eow))



2026-06-11T09:43:33.475631 [info     ] query_router: routed=TEMPORAL score=6.0 query='What should Veljko remember before EOW Friday 2026-06-12? Focus on unresolved obligations across Slack and email.' scores={'TEMPORAL': 6.0} [query_router]

2026-06-11T09:43:42.439208 [info     ] No events identified based on timestamp filtering, performing retrieval using triplet search on events and entities. [cognee.shared.logging_utils]

2026-06-11T09:43:43.349431 [info     ] Vector collection retrieval completed: Retrieved distances from 30 collections in 0.91s [cognee.shared.logging_utils]

2026-06-11T09:43:43.350113 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-11T09:43:43.356264 [info     ] ID-filtered retrieval: 102 nodes and 193 edges in 0.01s [cognee.shared.logging_utils]

2026-06-11T09:43:43.359547 [info     ] Graph projection completed: 102 nodes, 193 edges in 0.00s [CogneeGraph]

2026-06-11T09:43:43.360373 [info     ] Completed resolving edges to text [co

Unresolved items to handle before EOW Fri 2026-06-12:

1) Sign lease DocuSign packet — deadline: Fri 2026-06-12 at 12:00
source: Ana Petrovic emailed DocuSign packet and requested signature by 2026-06-12T12:00 (Ana → Veljko, 2026-06-10T14:37).

2) Reply to Jelena re: Friday dinner headcount — deadline (requested): EOD Thu 2026-06-11 (still needs a reply; restaurant needs final headcount)
source: Jelena Markovic email requesting RSVP by EOD 2026-06-11 (2026-06-10T19:24); Milica cc’d Veljko asking him to decide (2026-06-11T08:46).

3) Confirm Saturday climbing plan with Luka — deadline (requested): EOD Thu 2026-06-11 (to allow lane reservation; reply not recorded)
source: Luka message asking for RSVP by EOD 2026-06-11 (2026-06-11T07:55).


## See the complete typed graph

This final graph renders `TYPED_DATASET` after both Slack and email have been ingested with the custom `MemoryThread` / `Obligation` schema. It should contain the notebook-defined model types (`MemoryThread`, `Message`, `Person`, `Obligation`) rather than just Cognee's default `Entity` / `EntityType` nodes.


In [18]:
await show_graph("personal_memory_typed_graph.html", TYPED_DATASET)



2026-06-11T09:43:55.273556 [info     ] Retrieved 102 nodes and 193 edges in 0.01 seconds [cognee.shared.logging_utils]

2026-06-11T09:43:55.277265 [info     ] Graph visualization saved as /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/personal_memory_typed_graph.html [cognee.shared.logging_utils]

2026-06-11T09:43:55.277533 [info     ] The HTML file has been stored at path: /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/personal_memory_typed_graph.html [cognee.shared.logging_utils]

2026-06-11T09:43:55.305635 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]
